# Coupled network example: electrolyseres in Northern-Germany

Load a part of the German gas grid, from the SciGRID_gas dataset (www.gas.scigrid.de). This modified subset is somewhat related
to the envisioned hydrogen network, that some gas grid operators want to build from repurposed
natural gas pipelines.

Uncomment and execute the following cell if your are working on binder:

In [ ]:
#pip install plotly seaborn numba pyproj

In [ ]:
import pandapipes as ps
import pandapower as pp
import pandas as pd
from pandapipes.plotting import simple_plot as sp_gas
from pandapower.plotting import simple_plot as sp_power, simple_plotly, pf_res_plotly
from pandapower.plotting.collections import logger
logger.setLevel('ERROR')

%matplotlib inline

In [ ]:
gnet = ps.from_json('h2_backbone_NW_Germany_2GW_no_sources.json')


Set the gas to hydrogen and assume an operating pressure level of 30 bar.

For comparison, create a copy of the initial network.

Plot the grid to see how it looks like.

In [ ]:
# simple plot

In [ ]:
# previously created functions
from g3_plotting import create_pandapipes_collections, plot_gas_results
import matplotlib.pyplot as plt
import pandapipes.plotting as plot
collections = create_pandapipes_collections(gnet, show_junction_ID=True, label_size=10000)
plot.draw_collections(collections)
plt.show()

In [ ]:
plot_gas_results(gnet, junction_size=5000)
plt.show()

Load the German transmission grid (extra high voltage) from Simbench (www.simbench.de, `pip install simbench`).

In [ ]:
enet = pp.from_json('Simbench_EHV_Germany.json')

Plot the network.


In [ ]:
# simple plot

In [ ]:
# previously created functions
from e3_plotting import plot_lineloading
plot_lineloading(enet)

In [ ]:
# plotly for interactive plotting
_ = pf_res_plotly(enet, bus_size=3)

Prepare a multinet that contains both the power and the hydrogen grid.

In [ ]:
from pandapipes.multinet import create_empty_multinet, P2GControlMultiEnergy, add_net_to_multinet
...

Add 7 elektrolysers with a total of ~5 GW power consumption (700 MW each) and 70 % efficiency:
1. add the loads to the power grid at buses 2553, 2745, 2961, 1404, 2541, 2831, 2799
2. add the sources to the hydrogen grid at junctions 17, 53, 20, 32, 3, 8, 9
3. add P2G-controllers that connect the power loads and hydrogen sources

<img src="simbench_bus_IDs.png">

In [ ]:
...

In [ ]:
from pandapipes.multinet import P2GControlMultiEnergy
...

Run a coupled simulation.

In [ ]:
from pandapipes.multinet.control.run_control_multinet import run_control
...

Check the total sum of the produced hydrogen, the minimum pressure and the maximum velocity in
the hydrogen grid.

Plot the results of the initial hydrogen network and the hydrogen network with the electrolysers.

In [ ]:
plot_gas_results(gnet_initial, junction_size=5000)

In [ ]:
plot_gas_results(gnet, junction_size=5000)

Plot the line loading in the power grid.

In [ ]:
plot_lineloading(enet)

# Change fuel of gas turbines to hydrogen

Add the gas turbines to the line loading plotting function and plot the grid.


In [ ]:
# prepared plotting of different collections
def plot_lineloading_with_gasturbines(net, show_loadings=False):
    import pandapower.plotting as ppplot
    import numpy as np
    net = net.deepcopy()

    cmap_list=[(20, "green"), (50, "yellow"), (150, "red")]
    cmap, norm = ppplot.cmap_continuous(cmap_list)
    lc = ppplot.create_line_collection(net, net.line.index, zorder=1, cmap=cmap, norm=norm,
                                      linewidths=2)

    if show_loadings:
        loading_lst = net.res_line.loading_percent.tolist()  # list of all junction indices
        coords = zip(0.5*net.bus_geodata.x.loc[net.line.from_bus].values
                     + 0.5*net.bus_geodata.x.loc[net.line.to_bus].values,
                     0.5*net.bus_geodata.y.loc[net.line.from_bus].values
                     + 0.5*net.bus_geodata.y.loc[net.line.to_bus].values)
        # tuples of all junction coords

        loadingcol = ppplot.create_annotation_collection(size=0.15, texts=np.char.mod('%d',
                                                                                     loading_lst),
                                                        coords=coords,
                                                        zorder=150, color='k')
    else:
        loadingcol = None

    if hasattr(net, 'gen'):
        genc = ppplot.create_gen_collection(net, net.gen.loc[net.gen.type=='gas'].index, size=10000)
    else:
        genc = None

    ppplot.draw_collections([lc, loadingcol, genc], figsize=(8,6))
plot_lineloading_with_gasturbines(enet)

Find the ID of the four gas turbine generators in north-western Germany.

In [ ]:
span_x = enet.bus_geodata.x.max() - enet.bus_geodata.x.min()
span_y = enet.bus_geodata.y.max() - enet.bus_geodata.y.min()
nw_buses = enet.bus.loc[
    (enet.bus_geodata.x <= enet.bus_geodata.x.min() + 0.5*span_x) &
    (enet.bus_geodata.y >= enet.bus_geodata.y.min() + 0.75*span_y)].index
nw_gt = enet.gen.loc[(enet.gen.type=='gas') & enet.gen.bus.isin(nw_buses)]
# enet.bus_geodata.loc[nw_gt.bus].sort_values(by="x")

Connect the gas turbines (net.gen) 183, 181, 267 and 196 (from west to east) to the hydrogen grid.
Suitable junctions are 15, 13, 60, 8.

In [ ]:
from pandapipes.multinet import G2PControlMultiEnergy
...

Run a coupled simulation including the P2G and G2P units.